# Apriori from scratch on the episode transactions

**Pipeline position.** `episodes_v1.parquet` → **transactions** (one per episode, participants as items) → **frequent itemsets (this notebook)** → closed and maximal itemsets, rules → `recurring_cohorts.csv`.

**Why from scratch.** The goal is to see the mechanics: how many candidates each level generates, how many survive, and why the numbers explode when the minimum support falls. The library versions (`mlxtend`) come in the next notebook.

**Definitions**

- A **transaction** is the set of participants seen in one episode. An **itemset** $X$ is a set of participants.
- **Support count** $\sigma(X)$ is the number of transactions that contain every item of $X$. It counts distinct episodes, never slot rows (D13).
- **minsup** is a count of episodes here, not a fraction. $X$ is **frequent** if $\sigma(X)\ge \text{minsup}$.
- $C_k$ are the candidate itemsets of size $k$ and $L_k$ the frequent ones.
- **Downward closure:** if $X\subseteq Y$ then $\sigma(X)\ge\sigma(Y)$, so every subset of a frequent itemset is frequent.

**The algorithm**

1. Count single items and keep those with $\sigma\ge$ minsup: $L_1$.
2. **Join:** two frequent $(k-1)$-itemsets that share their first $k-2$ items make one $k$-candidate.
3. **Prune:** drop a candidate if any of its $(k-1)$-subsets is not in $L_{k-1}$.
4. **Count:** scan the transactions once and count every remaining candidate.
5. Keep the candidates with $\sigma\ge$ minsup: $L_k$. Stop when $L_k$ is empty.

Itemsets are stored as **sorted tuples**, so that the join can compare the first $k-2$ items.

In [ ]:
from itertools import combinations
from pathlib import Path
import random
import sys
import time
import pandas as pd

PROJECT_ROOT = Path.cwd().parent                   # notebooks/ is one level below the project root
PROCESSED = PROJECT_ROOT / "data" / "processed"

sys.path.append(str(PROJECT_ROOT))                 # so "from src..." works inside notebooks/
from src.features.transactions import build_transactions, support_count, transaction_stats

## 1. The functions

`count_items` is level 1. `make_candidates` is the join and the prune. `count_candidates` is the scan. `apriori` runs the levels and also records, for every level, how many candidates there were before and after pruning and how many were frequent.

In [ ]:
def count_items(transactions):
    """Level 1: in how many transactions does each single item appear?"""
    counts = {}
    for transaction in transactions:
        for item in transaction:
            counts[(item,)] = counts.get((item,), 0) + 1
    return counts


def make_candidates(frequent_prev, k):
    """Join and prune: build the size-k candidates from the sorted list of frequent (k-1)-itemsets."""
    known = set(frequent_prev)

    # join: same first k-2 items, and the last item of a is smaller than the last item of b
    joined = []
    for a in frequent_prev:
        for b in frequent_prev:
            if a[:-1] == b[:-1] and a[-1] < b[-1]:
                joined.append(a + (b[-1],))

    # prune: every (k-1)-subset of a candidate must be frequent
    candidates = []
    for candidate in joined:
        subsets = combinations(candidate, k - 1)
        if all(subset in known for subset in subsets):
            candidates.append(candidate)
    return joined, candidates


def count_candidates(transactions, candidates, k):
    """One scan: for every transaction, look at its k-subsets and count the ones that are candidates."""
    candidate_set = set(candidates)
    counts = {candidate: 0 for candidate in candidates}
    for transaction in transactions:
        if len(transaction) < k:
            continue                                # too short to contain a k-itemset
        for subset in combinations(transaction, k): # transactions are sorted, so subsets are sorted too
            if subset in candidate_set:
                counts[subset] += 1
    return counts


def apriori(transactions, minsup):
    """Return the list of frequent itemsets per level ({itemset: support}) and a table of candidate counts."""
    transactions = [tuple(sorted(t)) for t in transactions]

    item_counts = count_items(transactions)
    frequent = {itemset: n for itemset, n in item_counts.items() if n >= minsup}
    levels = [frequent]
    rows = [{"k": 1, "joined": len(item_counts), "candidates": len(item_counts), "frequent": len(frequent)}]

    k = 2
    while len(frequent) > 0:
        # an item that is in no frequent (k-1)-itemset cannot be in a frequent k-itemset, so drop it
        keep = set(item for itemset in frequent for item in itemset)
        transactions = [tuple(item for item in t if item in keep) for t in transactions]

        joined, candidates = make_candidates(sorted(frequent), k)
        if len(candidates) > 0:
            counts = count_candidates(transactions, candidates, k)
        else:
            counts = {}
        frequent = {itemset: n for itemset, n in counts.items() if n >= minsup}

        rows.append({"k": k, "joined": len(joined), "candidates": len(candidates), "frequent": len(frequent)})
        if len(frequent) > 0:
            levels.append(frequent)
        k += 1
    return levels, pd.DataFrame(rows)

## 2. Check on a toy example

Seven transactions over the items A to E, minsup = 3, small enough to do by hand. The expected frequent itemsets are A:5, B:6, C:5, AB:4, AC:4, BC:4 and ABC:3. D has $\sigma=2$ and E has $\sigma=1$, so they are not in $L_1$ and no candidate with D or E is ever generated.

In [ ]:
toy = [
    {"A", "B", "C"},        # T1
    {"A", "B", "C"},        # T2
    {"A", "B"},             # T3
    {"B", "C"},             # T4
    {"B", "D"},             # T5
    {"A", "B", "C", "D"},   # T6
    {"A", "C", "E"},        # T7
]
toy_levels, toy_table = apriori(toy, minsup=3)
print(toy_table.to_string(index=False))

expected = {("A",): 5, ("B",): 6, ("C",): 5,
            ("A", "B"): 4, ("A", "C"): 4, ("B", "C"): 4,
            ("A", "B", "C"): 3}
found = {itemset: n for level in toy_levels for itemset, n in level.items()}
print(found)
assert found == expected

## 3. The real transactions

One transaction per episode of `episodes_v1`. The length numbers matter: a transaction with $m$ items has $2^m-1$ non-empty subsets, so the few very long episodes dominate the cost at low minsup.

In [ ]:
episodes = pd.read_parquet(PROCESSED / "episodes_v1.parquet")
transactions = build_transactions(episodes)
stats = transaction_stats(transactions)
stats

## 4. Candidates per level at several minimum supports

minsup is a count of episodes. The table of each run shows, per level $k$: `joined` (candidates after the join), `candidates` (after the prune, these are the ones that get counted) and `frequent` (those with $\sigma\ge$ minsup).

The counting step gets expensive as minsup falls: a transaction with $m$ frequent items has $\binom{m}{k}$ subsets of size $k$, and the longest episodes have more than a hundred participants. In a test run, minsup = 100 took several seconds and minsup = 75 did not finish within 45 seconds, so the list stops at 100.

In [ ]:
MINSUPS = [400, 300, 250, 200, 150, 100]            # counts of episodes, high to low

summary = []
tables = {}
results = {}
for minsup in MINSUPS:
    start = time.time()
    levels, table = apriori(transactions, minsup)
    seconds = time.time() - start

    tables[minsup] = table
    results[minsup] = levels
    summary.append({
        "minsup": minsup,
        "minsup_share_pct": round(100 * minsup / len(transactions), 2),
        "counted_candidates": int(table["candidates"].sum()),
        "frequent_itemsets": int(table["frequent"].sum()),
        "longest_itemset": len(levels),                # levels only holds non-empty levels
        "seconds": round(seconds, 1),
    })
    print(f"minsup = {minsup}")
    print(table.to_string(index=False))
    print()

pd.DataFrame(summary)

## 5. Checks

Two checks against the definitions. First, a random sample of the found itemsets is recounted with `support_count`, the slow definition from `transactions.py`. Second, at level 2 the prune can remove nothing: every 1-subset of a pair is frequent by construction, so the number of counted pairs must be exactly $\binom{|L_1|}{2}$.

In [ ]:
random.seed(0)
for minsup in MINSUPS:
    found = [(itemset, n) for level in results[minsup] for itemset, n in level.items()]
    for itemset, n in random.sample(found, min(30, len(found))):
        assert support_count(transactions, itemset) == n
print("sampled supports agree with the definition")

for minsup in MINSUPS:
    table = tables[minsup]
    n_l1 = int(table.loc[table["k"] == 1, "frequent"].iloc[0])
    n_c2 = int(table.loc[table["k"] == 2, "candidates"].iloc[0])
    assert n_c2 == n_l1 * (n_l1 - 1) // 2
    print(f"minsup {minsup}: |L1| = {n_l1}, |C2| = {n_c2} = C({n_l1}, 2)")

## 6. Questions to answer before moving on

1. Compare `joined` and `candidates` at $k=3$ in each run. When does the prune remove something, and why never at $k=2$?
2. How does the number of counted candidates change from minsup 400 to 250, and how does the number of frequent itemsets change? What does the ratio say about the work spent on candidates that turn out to be infrequent?
3. Itemsets here are groups of participants that share an episode. What would you have to check before calling a frequent itemset of three or more a cohort (think about what a transaction is, and what $\sigma$ counts)?